In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, 6ade8dfa-f2a8-404f-838a-809d506fb2b7, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, 6ade8dfa-f2a8-404f-838a-809d506fb2b7, 4, Finished, Available, Finished, False)

18 projects found


In [3]:
all_submittals = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling submittals for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            f"https://api.procore.com/rest/v1.1/projects/{project_id}/submittals",
            headers=headers,
            params={
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_submittals.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total submittals: {len(all_submittals)}")

StatementMeta(, 6ade8dfa-f2a8-404f-838a-809d506fb2b7, 5, Finished, Available, Finished, False)

Pulling submittals for: 1100 Fulton Street
Pulling submittals for: 11 ESSEX ST
Pulling submittals for: 337A & 337B West Broadway Rehabilitaion Work
Pulling submittals for: 360 Lexington 8th & 20th Floor
Pulling submittals for: 549 Munroe Av
Pulling submittals for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling submittals for: Boys & Girls Club
Pulling submittals for: EMBANKMENT PHASE II
Pulling submittals for: Embankment Phase III
Pulling submittals for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling submittals for: Lillipvt 45 Renwick St
Pulling submittals for: PCNA 711 11TH AVE
Pulling submittals for: Sandbox Test Project
Pulling submittals for: SaunaLounge 45 South 3 Street, Brooklyn, NY
Pulling submittals for: Standard Project Template
Pulling submittals for: SYMRISE - 15th & 16th Flr
Pulling submittals for: TEST - ABM SUBORDINATE
Pulling submittals for: VOCO HOTEL TSQ
Done! Total submittals: 2242


In [4]:
import pandas as pd
import re

clean_rows = []
for row in all_submittals:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_submittals_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_submittals_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, 6ade8dfa-f2a8-404f-838a-809d506fb2b7, 6, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully
